In [1]:
from yt_dlp import YoutubeDL
import torch
import torch.nn.functional as F
import cv2
import time

url = "https://www.youtube.com/watch?v=lLY46UGteJ0"  # 【玉山國家公園】熊鷹小夫妻－夫(右)妻(左-伊布)
ydl_opts = {
    'format': 'best',
    'js-runtimes': ['node'],
    'remote-components': 'ejs:github',
}

with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    video_url = info['url']
    YouTubeVideo = cv2.VideoCapture(video_url)
    localTestVideo = cv2.VideoCapture('homework_1_test_video.mp4')

    if not YouTubeVideo.isOpened() or not localTestVideo.isOpened():
        print('cannot open youtube or local test video')
        YouTubeVideo.release()
        localTestVideo.release()
    else:
        fps = int(YouTubeVideo.get(cv2.CAP_PROP_FPS))
        if fps <= 0:
            fps = 30
        #w=640,h=360
        width = int(YouTubeVideo.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(YouTubeVideo.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f'width:{width}, height: {height}')

        # 每個子畫面的尺寸，2x4 拼接後比較容易完整顯示
        tile_width = max(width //2 , 1)
        tile_height = max(height , 1)


        frame_num = 0
        play_flag = 1
        #馬賽克的程度，數值越大越模糊
        mosaic_level = 16
        total_frame = int(YouTubeVideo.get(cv2.CAP_PROP_FRAME_COUNT))
        childWindowRow=2
        childWindowColumn=4
        # 存檔設定
        output_filename = 'hw1_output.mp4'
        output_width = tile_width * childWindowColumn
        output_height = tile_height * childWindowRow
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_filename, fourcc, fps, (output_width, output_height))
        if not out.isOpened():
            print('cannot open video writer, output will not be saved')
        else:
            print(f'video writer opened:{output_filename},size=({output_width},{output_height}),fps={fps}')

        def set_frame_number(x):  # 進度條
            global frame_num, play_flag
            if play_flag == 0:  # 暫停才能調整進度條
                frame_num = x
                YouTubeVideo.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                localTestVideo.set(cv2.CAP_PROP_POS_FRAMES, frame_num)


        def play(x):
            global play_flag
            play_flag = x


        cv2.namedWindow('hw1')
        cv2.createTrackbar('frame no.', 'hw1', 0, max(total_frame - 1, 1), set_frame_number)
        cv2.createTrackbar('play', 'hw1', 0, 1, play)
        cv2.setTrackbarPos('play', 'hw1', play_flag)
        frame_yt = None
        frame_local = None
        written_frames = 0

        while True:
            cv2.setTrackbarPos('frame no.', 'hw1', frame_num)
            t1 = time.perf_counter_ns()
            if play_flag:
                grabbed_yt, new_frame_yt = YouTubeVideo.read()
                grabbed_local, new_frame_local = localTestVideo.read()
                if not grabbed_yt or not grabbed_local:
                    print('video ended')
                    break

                frame_yt = new_frame_yt
                frame_local = new_frame_local
                frame_num += 1
                #影像負片
                frame_yt_Negative_image=255-frame_yt
                #histogram equalization
                frame_yt_ycrcb=cv2.cvtColor(frame_yt, cv2.COLOR_BGR2YCrCb)
                channels=cv2.split(frame_yt_ycrcb)
                #channels[0]是亮度通道，對它做直方圖均衡化
                cv2.equalizeHist(channels[0], channels[0])
                frame_yt_equalized = cv2.merge(channels)
                frame_yt_equalized=cv2.cvtColor(frame_yt_equalized, cv2.COLOR_YCrCb2BGR)
                #gaussian blur,kernel size 15*15
                frame_yt_gaussian_blur=cv2.GaussianBlur(frame_yt,(15,15),0)

                #hsv
                frame_local_hsv=cv2.cvtColor(frame_local, cv2.COLOR_BGR2HSV)

                #用pytorch做馬賽克,opencv是(height,weight,channels),pytorch是(channels,height,weight)
                h, w = frame_local.shape[:2]
                #permute(2, 0, 1)將channels移到第一維，unsqueeze(0)增加一個batch維度，最後除以255.0將像素值統一到0,1
                tensor_input = torch.from_numpy(frame_local).float().permute(2, 0, 1).unsqueeze(0) / 255.0
                small_h = max(1, h // mosaic_level)
                small_w = max(1, w // mosaic_level)
                #把影像縮小到(small_h, small_w)，再放大回原來的大小，這樣就會有馬賽克效果
                small = F.interpolate(tensor_input, size=(small_h, small_w), mode='bilinear', align_corners=False)
                #放大回原來的大小，mode='nearest'會讓每個小區塊都變成同一個顏色
                mosaic = F.interpolate(small, size=(h, w), mode='nearest')
                #squeeze(0)把batch維度去掉，permute(1, 2, 0)把channels移回最後一維，
                #clamp(0, 1)確保像素值在0到1之間，最後乘以255轉回0-255的範圍，並轉成uint8格式
                frame_local_mosaic = (mosaic.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype('uint8')
                
                #用pytorch做edge detection
                #sobel kernel,用來計算x和y方向的梯度
                sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=tensor_input.dtype,
                                       device=tensor_input.device).view(1, 1, 3, 3).repeat(3, 1, 1, 1)
                sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=tensor_input.dtype, 
                                       device=tensor_input.device).view(1, 1, 3, 3).repeat(3, 1, 1, 1)
                #groups=3表示對3個channel分別做卷積
                gradientX=F.conv2d(tensor_input, sobel_x, padding=1, groups=3)
                gradientY=F.conv2d(tensor_input, sobel_y, padding=1, groups=3)
                #這裡的gradient是每個像素的梯度強度，可以用來表示邊緣的強度
                gradient = torch.sqrt(gradientX ** 2 + gradientY ** 2)
                frame_local_edge = (gradient.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype('uint8')
                
                display_yt = cv2.resize(frame_yt, (tile_width, tile_height))
                display_eq = cv2.resize(frame_yt_equalized, (tile_width, tile_height))
                display_neg = cv2.resize(frame_yt_Negative_image, (tile_width, tile_height))
                display_yt_gaussian_blur = cv2.resize(frame_yt_gaussian_blur, (tile_width, tile_height))
                display_local = cv2.resize(frame_local, (tile_width, tile_height))
                display_local_hsv = cv2.resize(frame_local_hsv, (tile_width, tile_height))
                display_local_mosaic = cv2.resize(frame_local_mosaic, (tile_width, tile_height))
                display_local_edge = cv2.resize(frame_local_edge, (tile_width, tile_height))

                # 上面四格：YouTube 影片（原圖、直方圖均衡化、負片、高斯模糊）
                top_row = cv2.hconcat([display_yt, display_eq, display_neg, display_yt_gaussian_blur])
                # 下面三格：本地測試影片（原圖、HSV、馬賽克）
                bottom_row = cv2.hconcat([display_local, display_local_hsv, display_local_mosaic, display_local_edge])
                # 上下拼接成 2x4
                combined_frame = cv2.vconcat([top_row, bottom_row])
                cv2.imshow('hw1', combined_frame)

                # 寫入輸出檔
                if out.isOpened():
                    out.write(combined_frame)
                    written_frames += 1
                    if written_frames % 120 == 0:
                        print(f'written_frames={written_frames}')


            t2 = 1000 // fps - (time.perf_counter_ns() - t1) // 1000000
            k = cv2.waitKey(max(t2, 1))


            if k == 27:  # ESC
                break

        cv2.destroyAllWindows()
        if out.isOpened():
            out.release()
            print(f'saved to {output_filename}, total written_frames={written_frames}')
        YouTubeVideo.release()
        localTestVideo.release()

[youtube] Extracting URL: https://www.youtube.com/watch?v=lLY46UGteJ0
[youtube] lLY46UGteJ0: Downloading webpage


[youtube] lLY46UGteJ0: Downloading android vr player API JSON
width:640, height: 360
video writer opened:hw1_output.mp4,size=(1280,720),fps=30
written_frames=120
video ended
saved to hw1_output.mp4, total written_frames=201


In [16]:
pip install gtts

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pyttsx3 
from moviepy import AudioFileClip
from moviepy import VideoFileClip, TextClip, CompositeVideoClip, CompositeAudioClip, concatenate_audioclips
from moviepy.video.tools.subtitles import SubtitlesClip
from moviepy.video.fx import MultiplySpeed
from moviepy.audio.fx import MultiplyVolume

class TexttoSpeech:
    def __init__(self):
        # Initialize the engine       
        self.engine = pyttsx3.init()
        voices = self.engine.getProperty('voices')
        self.chinese_voice_id = None

        for v in voices:
            if "Chinese" in v.name or "ZH-TW" in v.id:
                self.chinese_voice_id = v.id
                break

        if self.chinese_voice_id:
            self.engine.setProperty('voice', self.chinese_voice_id)
        else:
            print("No Chinese voice found. Please install one in System Settings.")

    def text_to_speech(self,message):
        self.engine.say(message) 
    
    def text_to_mp3(self,message,mp3file):
        self.engine.save_to_file(message, mp3file)

    def runAndWait(self):
        self.engine.runAndWait()
  
ts = TexttoSpeech()

#腳本，格式是一行一句,注意尾巴不要留空白。
subtitles = '''    這是作業一\n
    上面第一個row由左到右展示了原始的YouTube影片\n
    直方圖均衡化\n
    影像負片\n
    高斯模糊,kernel size是15乘15\n
    第二個row由左到右展示了原始的本地測試影片\n
    將BGR轉換到HSV色彩空間\n
    用pytorch做的馬賽克效果\n
    用pytorch做的edge detection\n
    '''
   
# 來源視訊，背景音樂，目標視訊檔名。    
source_video_filename     = "hw1_output.mp4" 
background_music_filename = "calm-background-for-video-121519.mp3" # google this mp3 and download it by yourself
target_video_with_subtitle= "homework_1_subtitle.mp4"

lines = [msg.strip() for msg in subtitles.split('\n') if msg.strip()]
for i, msg in enumerate(lines):
    print(f'line {i}: {msg}')
total_lines = len(lines)
speech= []
#念每一句旁白。
for i,msg in enumerate(lines):
    print(f'generate audio clip for {msg} ({i+1}/{total_lines})')
    ts.text_to_mp3(msg,'voice-msg-{:d}.mp3'.format(i))
#ts.engine.runAndWait()#這行有時候會卡住

### 根據每句旁白時間長度，計算每句旁白在視訊裡的起始與結束時間
speech = [AudioFileClip('voice-msg-{:d}.mp3'.format(i)) for i, _ in enumerate(lines)]   
duration = np.array([0]+[s.duration for s in speech])   
cumduration = np.cumsum(duration)
total_duration = int((cumduration[-1]+7)/8)*8    
print(total_duration)

clip = VideoFileClip(source_video_filename)

#調整目標視訊播放速度，使得目標視訊播放時間比念完全部旁白長一點。
speed_factor = clip.duration / total_duration
clip = clip.with_effects([MultiplySpeed(speed_factor)])

# 產生旁白字幕，注意msjh.ttc字型檔要在這個程式所在目錄。
generator = lambda txt: TextClip(text=txt, font='msjh.ttc', font_size=36, color='white',
                                 bg_color=(10,10,10,150), method='caption',size=(clip.w, 64))
subs_data = [((cumduration[i],cumduration[i+1]),s) for i,s in enumerate(lines)]
subtitles = SubtitlesClip(subs_data, make_textclip=generator)

#產生有字幕的目標視訊。
final_clip = CompositeVideoClip([clip, subtitles.with_position(("center","bottom"))])

#背景音樂，只擷取目標視訊長度片段，並將音量調小。
background_audio = AudioFileClip(background_music_filename)
baudio1 = background_audio.subclipped(background_audio.duration-total_duration).with_effects([MultiplyVolume(0.1)])

#將目標視訊的音訊設為混合來源視訊音訊、背景音樂、旁白音訊的音訊。
final_clip = final_clip.with_audio(CompositeAudioClip([baudio1,concatenate_audioclips(speech)]))

#輸出至目標視訊檔。
final_clip.write_videofile(target_video_with_subtitle)



line 0: 這是作業一
line 1: 上面第一個row由左到右展示了原始的YouTube影片
line 2: 直方圖均衡化
line 3: 影像負片
line 4: 高斯模糊,kernel size是15乘15
line 5: 第二個row由左到右展示了原始的本地測試影片
line 6: 將BGR轉換到HSV色彩空間
line 7: 用pytorch做的馬賽克效果
line 8: 用pytorch做的edge detection
generate audio clip for 這是作業一 (1/9)
generate audio clip for 上面第一個row由左到右展示了原始的YouTube影片 (2/9)
generate audio clip for 直方圖均衡化 (3/9)
generate audio clip for 影像負片 (4/9)
generate audio clip for 高斯模糊,kernel size是15乘15 (5/9)
generate audio clip for 第二個row由左到右展示了原始的本地測試影片 (6/9)
generate audio clip for 將BGR轉換到HSV色彩空間 (7/9)
generate audio clip for 用pytorch做的馬賽克效果 (8/9)
generate audio clip for 用pytorch做的edge detection (9/9)
24
MoviePy - Building video homework_1_subtitle.mp4.
MoviePy - Writing audio in homework_1_subtitleTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video homework_1_subtitle.mp4



MoviePy - Done !
MoviePy - video ready homework_1_subtitle.mp4
